# Trial Inspector

Set one `generation_run_id` + `trial_id` below and re-run every cell (Run All) to see
everything about that trial in one place: the canonical scene/character data, the U/S/P
dialogue transcripts, the H1 judge prompts (with a check for whether they're really
condition-blind), the full H1 metric breakdown per condition, and the H2 realism
judge's critique, scores, and score explanation per condition.

This is a drill-down tool for one trial at a time, separate from `results_dashboard.ipynb`
(which aggregates across trials/conditions).

In [190]:
import glob
import json
from pathlib import Path

REPO_ROOT = Path.cwd().parent  # this notebook lives in notebooks/, one level under the repo root
H1_PATH = REPO_ROOT / "output" / "evaluation" / "h1_consistency.jsonl"
H2_PATH = REPO_ROOT / "output" / "evaluation" / "h2_realism.jsonl"
TRAJECTORY_GLOB = str(REPO_ROOT / "output" / "record" / "*" / "trajectory" / "*.jsonl")
ALL_ACTIONS_GLOB = str(REPO_ROOT / "output" / "record" / "*" / "all" / "*.jsonl")

CONDITION_ORDER = ["U", "S", "P"]


def load_jsonl(path):
    with open(path, encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]

In [191]:
def find_trajectory_records(generation_run_id, trial_id):
    """Search every trajectory file under output/record/*/trajectory/, not one hardcoded
    path, so this keeps working if more scenes/titles/models get added later."""
    matches = {}
    for path in sorted(glob.glob(TRAJECTORY_GLOB)):
        for record in load_jsonl(path):
            if record.get("generation_run_id") == generation_run_id and record.get("trial_id") == trial_id:
                matches[record["context_condition"]] = record
    return matches


def find_generation_calls(generation_run_id, trial_id, condition):
    """One line in the `all` record file per (trial, condition), saved once the whole
    trial finishes; its `record` list holds one {prompt, response, duration} dict per
    round, in round order (record[0] = round 1). Search every `all` file, not one
    hardcoded path, for the same reason as find_trajectory_records()."""
    for path in sorted(glob.glob(ALL_ACTIONS_GLOB)):
        for line in load_jsonl(path):
            if (
                line.get("generation_run_id") == generation_run_id
                and line.get("trial_id") == trial_id
                and line.get("context_condition") == condition
            ):
                return line["record"]
    return None


def find_h1_rows(generation_run_id, trial_id):
    by_condition = {condition: [] for condition in CONDITION_ORDER}
    for row in load_jsonl(H1_PATH):
        if row.get("generation_run_id") == generation_run_id and row.get("trial_id") == trial_id:
            by_condition[row["context_condition"]].append(row)
    for condition in by_condition:
        by_condition[condition].sort(key=lambda r: r["round"])
    return by_condition


def find_h2_rows(generation_run_id, trial_id):
    """One H2 row per (trial, condition): H2 scores the whole trajectory at once, not
    per round. Returns None for a condition with no H2 row yet."""
    by_condition = {condition: None for condition in CONDITION_ORDER}
    if not H2_PATH.exists():
        return by_condition
    for row in load_jsonl(H2_PATH):
        if row.get("generation_run_id") == generation_run_id and row.get("trial_id") == trial_id:
            by_condition[row["context_condition"]] = row
    return by_condition


## Set the trial to inspect

In [192]:
GENERATION_RUN_ID = "cb_hp_019_luna_neville-1790690383028011000"
TRIAL_ID = "cb_hp_019_luna_neville_t11"

trajectory_records = find_trajectory_records(GENERATION_RUN_ID, TRIAL_ID)
h1_rows = find_h1_rows(GENERATION_RUN_ID, TRIAL_ID)
generation_calls = {c: find_generation_calls(GENERATION_RUN_ID, TRIAL_ID, c) for c in CONDITION_ORDER}
h2_rows = find_h2_rows(GENERATION_RUN_ID, TRIAL_ID)

missing_trajectory = [c for c in CONDITION_ORDER if c not in trajectory_records]
missing_h1 = [c for c in CONDITION_ORDER if not h1_rows[c]]
missing_generation = [c for c in CONDITION_ORDER if not generation_calls[c]]
if missing_trajectory:
    raise ValueError(f"No trajectory record found for conditions: {missing_trajectory}")
if missing_h1:
    raise ValueError(f"No H1 rows found for conditions: {missing_h1}")
if missing_generation:
    raise ValueError(f"No generation-call trace ('all' record) found for conditions: {missing_generation}")

print(f"Found trial {TRIAL_ID!r} in run {GENERATION_RUN_ID!r}: all of U/S/P present.")
print({c: len(h1_rows[c]) for c in CONDITION_ORDER}, "rounds scored per condition")
missing_h2 = [c for c in CONDITION_ORDER if h2_rows[c] is None]
if missing_h2:
    # Not an error: H2 sections 9-11 just report the gap, the H1 sections still work.
    print(f"No H2 row yet for conditions: {missing_h2}")


Found trial 'cb_hp_019_luna_neville_t11' in run 'cb_hp_019_luna_neville-1790690383028011000': all of U/S/P present.
{'U': 10, 'S': 10, 'P': 10} rounds scored per condition


### 1. Scene information

In [193]:
reference = trajectory_records["U"]
scene = reference["scene"]
player = reference["player"]
target = reference["target"]

print("[Scene]")
for key in ("event", "time", "location", "description"):
    print(f"{key.capitalize()}: {scene.get(key)}")

print()
print(f"[Player — {player['name']}]")
for key in ("position", "states", "description"):
    print(f"{key.capitalize()}: {player.get(key)}")

print()
print(f"[Target — {target['name']}]")
for key in ("position", "states", "description", "self_belief", "environment_belief"):
    print(f"{key.capitalize()}: {target.get(key)}")

print()
print("[Action Reference]")
print(reference["action_reference"])

# docs/research-contract.md §5: fields-and-values(U) = fields-and-values(S) = fields-and-values(P).
# Cheap, concrete check that this trial actually respects it, not just an assumption.
equivalent = all(
    (trajectory_records[c]["scene"], trajectory_records[c]["target"], trajectory_records[c]["action_reference"])
    == (scene, target, reference["action_reference"])
    for c in CONDITION_ORDER
)
print()
print(f"Scene/character/action_reference identical across U/S/P: {equivalent}")

[Scene]
Event: The Secret Meeting
Time: Late evening
Location: The Shrieking Shack, Hogwarts
Description: The atmosphere is eerie as the wind howls around the Shrieking Shack. The moon is hidden behind clouds, casting a mysterious glow over the scene. A secret meeting is about to take place between two key characters.

[Player — Luna Lovegood]
Position: Standing near the entrance of the Shrieking Shack, looking around warily.
States: Luna is feeling a mix of anticipation and caution, eager to discuss the latest developments but aware of the risks involved in meeting in such a secluded place.
Description: Luna is a quirky and insightful student, known for her unique perspective on the magical world. She is wearing her characteristic radish earrings and is excited about the upcoming meeting.

[Target — Neville Longbottom]
Position: Hiding behind a large rock, keeping an eye out for any sign of trouble.
States: Neville is nervous but determined, ready to support Luna and the plan they hav

In [194]:
def format_interaction(record):
    player_name = record["player"]["name"]
    target_name = record["target"]["name"]
    lines = []
    for turn in record["turns"]:
        lines.append(f"Round {turn['round']}")
        lines.append(f"{player_name}: {turn['player_dialogue']}")
        lines.append(f"{target_name}: {turn['npc_dialogue']}")
    return "\n".join(lines)


def print_generation_prompt_structure(condition):
    """Round 1's actual filled generation prompt for this condition -- this is where
    U/S/P genuinely differ in structure (unlike the H1 judge prompt, which is
    deliberately condition-blind). Later rounds follow the same structure with the
    rolling Observation/memory content updated each round; not reprinted per round
    here since the structure itself doesn't change, only that one field's content."""
    first_call = generation_calls[condition][0]
    print(first_call["prompt"])

### 2. U — interaction

**Prompt structure used to generate U's responses** (round 1, representative):

In [195]:
print_generation_prompt_structure("U")

Please act as Neville Longbottom and respond using the tone, manner and vocabulary Neville Longbottom would use. Use the following context to produce one appropriate line of dialogue.
The Secret Meeting
Late evening
The Shrieking Shack, Hogwarts
The atmosphere is eerie as the wind howls around the Shrieking Shack. The moon is hidden behind clouds, casting a mysterious glow over the scene. A secret meeting is about to take place between two key characters.
Neville Longbottom
Neville is a brave and loyal friend to Harry Potter and his classmates. He has been practicing his magical skills and is ready to contribute to their cause.
Hiding behind a large rock, keeping an eye out for any sign of trouble.
Neville is nervous but determined, ready to support Luna and the plan they have put in place.



Previous interaction:
Luna Lovegood:Luna produces a charm to reveal any hidden enchantments around the Shack, ensuring their meeting place is secure.;Luna Lovegood:“REVELLOUS!”
Neville Longbottom

In [196]:
print(format_interaction(trajectory_records["U"]))

Round 1
Luna Lovegood: Harry sent word, he'll meet us at the Hog's Head at midnight.
Neville Longbottom: Right, but we should be careful. The last thing we need is to get caught out here. Let’s make sure we’re ready for anything.
Round 2
Luna Lovegood: Good thinking. And when we get there, the signal is three taps on the window.
Neville Longbottom: Right, but let’s keep our voices down—remember what happened last time we weren’t careful? I’d rather not end up in another close call.
Round 3
Luna Lovegood: How did you get out of the castle without anyone seeing you?
Neville Longbottom: I, um, just followed the secret passage from the Herbology shed. But we really need to be careful—if Filch catches us, I don’t think even a good potion could save us this time!
Round 4
Luna Lovegood: Remind me, where and when are we meeting Harry?
Neville Longbottom: We’re meeting him at the Hog's Head at midnight, but we really need to be quiet—if anyone hears us, it could ruin everything. I don’t want to

### 3. S — interaction

**Prompt structure used to generate S's responses** (round 1, representative):

In [197]:
print_generation_prompt_structure("S")

Please act as Neville Longbottom  and respond using the tone, manner and vocabulary Neville Longbottom would use.
 The context and Neville Longbottom's information is as follows:
Event: The Secret Meeting
Time: Late evening
Location: The Shrieking Shack, Hogwarts
Description: The atmosphere is eerie as the wind howls around the Shrieking Shack. The moon is hidden behind clouds, casting a mysterious glow over the scene. A secret meeting is about to take place between two key characters.
Character Profile:
Name: Neville Longbottom
Character Description: Neville is a brave and loyal friend to Harry Potter and his classmates. He has been practicing his magical skills and is ready to contribute to their cause.
Position: Hiding behind a large rock, keeping an eye out for any sign of trouble.
Current State: Neville is nervous but determined, ready to support Luna and the plan they have put in place.
Self Belief: 
Environment Belief: 
Recent Memory(Ordered in reverse chronological order): 
Obs

In [198]:
print(format_interaction(trajectory_records["S"]))

Round 1
Luna Lovegood: Harry sent word, he'll meet us at the Hog's Head at midnight.
Neville Longbottom: Right, but let’s make sure we’re ready for anything. I’ve got my gillyweed and a few other bits just in case. We can’t afford to be caught off guard, not tonight.
Round 2
Luna Lovegood: Good thinking. And when we get there, the signal is three taps on the window.
Neville Longbottom: Right, Luna, just remember to keep your voice down. This place gives me the creeps, and we don’t want anyone overhearing us. If something goes wrong, we need to be ready to act fast. Let’s stick to the plan and watch each other’s backs, yeah?
Round 3
Luna Lovegood: How did you get out of the castle without anyone seeing you?
Neville Longbottom: I, um, used the Invisibility Cloak. It’s a bit risky, but I figured it was worth it to make sure we could meet without anyone noticing. Just… keep your head down, alright? We can’t let anyone see us here. This place is already spooky enough without drawing attenti

### 4. P — interaction

**Prompt structure used to generate P's responses** (round 1, representative):

In [199]:
print_generation_prompt_structure("P")

Please act as Neville Longbottom and respond using the tone, manner and vocabulary Neville Longbottom would use.
The context below is ordered from most important (Priority 1) to least important. Use all relevant context when it is compatible. If two pieces of context conflict, follow the one with the smaller priority number.
Priority 1 — current_world_constraints
Instruction: Must not be contradicted, and overrides all other priorities.
Time: Late evening
Location: The Shrieking Shack, Hogwarts
Position: Hiding behind a large rock, keeping an eye out for any sign of trouble.
Priority 2 — world_background
Instruction: Keep the response consistent with the fictional world.
Event: The Secret Meeting
Description: The atmosphere is eerie as the wind howls around the Shrieking Shack. The moon is hidden behind clouds, casting a mysterious glow over the scene. A secret meeting is about to take place between two key characters.
Priority 3 — character_guidance
Instruction: Use these fields to pr

In [200]:
print(format_interaction(trajectory_records["P"]))

Round 1
Luna Lovegood: Harry sent word, he'll meet us at the Hog's Head at midnight.
Neville Longbottom: Midnight? That’s... that’s not too far off. We should make sure we’re ready for anything. I just hope everything goes smoothly. I mean, it’s Harry, but still… we can’t be too careful.
Round 2
Luna Lovegood: Good thinking. And when we get there, the signal is three taps on the window.
Neville Longbottom: Right, three taps on the window. Let’s just make sure we’re quiet when we get there. I mean, it’s the Shrieking Shack, and who knows what could be lurking around here. We can’t let anything go wrong, especially not with Harry counting on us. Just... stay alert, okay?
Round 3
Luna Lovegood: How did you get out of the castle without anyone seeing you?
Neville Longbottom: Oh, well, I just used a little bit of the Invisibility Cloak. It’s not like I’m the best at sneaking around, but I thought it might help. I really didn’t want to get caught, especially with everything going on. I mean,

### 5. Judge prompt (H1)

One prompt per round per condition. `[Scene]`/`[Character]`/`[Action Reference]` should be
identical across U/S/P (same values, per the equivalence check above); `[Prior turns]`/
`[Current turn]` are expected to differ, since that's the actual generated dialogue, which
does vary by condition. The check below only compares the part *before* `[Prior turns]`.

In [201]:
def prompt_prefix(prompt_text):
    marker = "[Prior turns]"
    index = prompt_text.find(marker)
    return prompt_text[:index] if index != -1 else prompt_text


rounds = [row["round"] for row in h1_rows["U"]]
for round_number in rounds:
    prompts = {
        condition: next(row["prompt"] for row in h1_rows[condition] if row["round"] == round_number)
        for condition in CONDITION_ORDER
    }
    prefixes = {condition: prompt_prefix(text) for condition, text in prompts.items()}
    identical = len(set(prefixes.values())) == 1

    print(f"===== Round {round_number} — scene/character/rubric identical across U/S/P: {identical} =====")
    for condition in CONDITION_ORDER:
        print(f"--- {condition} ---")
        print(prompts[condition])
        print()

===== Round 1 — scene/character/rubric identical across U/S/P: True =====
--- U ---
Please evaluate one turn of Neville Longbottom's dialogue against the supplied scene and character information, and against the character's own earlier turns in this conversation.

[Scene]
Event: The Secret Meeting
Time: Late evening
Location: The Shrieking Shack, Hogwarts
Description: The atmosphere is eerie as the wind howls around the Shrieking Shack. The moon is hidden behind clouds, casting a mysterious glow over the scene. A secret meeting is about to take place between two key characters.

[Character — Neville Longbottom]
Position: Hiding behind a large rock, keeping an eye out for any sign of trouble.
Current State: Neville is nervous but determined, ready to support Luna and the plan they have put in place.
Description: Neville is a brave and loyal friend to Harry Potter and his classmates. He has been practicing his magical skills and is ready to contribute to their cause.

[Action Reference]


### 6. U — H1 metrics

In [202]:
def format_world_consistency(entry):
    """world_consistency has no top-level evidence field -- its decision is derived
    (in evaluate.py) from a claims list, so show that list rather than a single quote.
    "checked_against" is required on every claim now (shown as "conflicts with" for an
    inconsistent claim, "checked against" for a consistent one, including the literal
    "No supplied fact addresses this claim." placeholder when nothing applies). See the
    2026-09-16 provenance entries (claim-list redesign, checked_against rename, then
    required-on-every-claim).

    Each claim also carries "quote" (the verbatim response fragment it's drawn from)
    and "quote_verified" (whether that fragment actually appears in the NPC's real
    dialogue) -- a False here means the judge's "claim" may be misrepresenting what was
    actually said, worth checking by hand. See the 2026-09-23 provenance entry."""
    lines = [f"World Consistency: {entry['decision']}"]
    if not entry["claims"]:
        lines.append("  (no checkable claims)")
    for claim in entry["claims"]:
        verified_flag = "" if claim.get("quote_verified", True) else "  [UNVERIFIED QUOTE]"
        if claim["consistent"]:
            lines.append(f"  [consistent] {claim['claim']}{verified_flag}")
            lines.append(f"    checked against: {claim['checked_against']}")
        else:
            lines.append(f"  [CONTRADICTS] {claim['claim']}{verified_flag}")
            lines.append(f"    conflicts with: {claim['checked_against']}")
        lines.append(f"    quote: {claim.get('quote', '(none)')!r}")
    return "\n".join(lines)


def format_h1_metrics(rows):
    blocks = []
    for row in rows:
        decisions = row["decisions"]
        context_use = decisions["context_use"]
        no_contradiction = decisions["no_contradiction"]
        lines = [
            f"Round {row['round']}",
            f"Context Use: {context_use['decision']}",
            f"{context_use['evidence'] or '(none — N/A)'}",
            format_world_consistency(decisions["world_consistency"]),
            f"No Contradiction: {no_contradiction['decision']}",
            f"Contradicted round: {no_contradiction['contradicted_round'] if no_contradiction['contradicted_round'] is not None else '(none)'}",
            f"{no_contradiction['evidence'] or '(none — N/A)'}",
        ]
        blocks.append("\n".join(lines))
    return "\n\n".join(blocks)


print(format_h1_metrics(h1_rows["U"]))

Round 1
Context Use: PASS
Elicited fact: Neville is nervous but determined, ready to support Luna and the plan they have put in place. Response: 'we should be careful. The last thing we need is to get caught out here. Let’s make sure we’re ready for anything.'
World Consistency: PASS
  [consistent] they should be cautious
    checked against: Character: Neville is nervous but determined, ready to support Luna and the plan they have put in place.
    quote: 'we should be careful'
  [consistent] they do not want to get caught
    checked against: Character: Neville is nervous but determined, ready to support Luna and the plan they have put in place.
    quote: 'The last thing we need is to get caught out here'
  [consistent] they should be prepared for any situation
    checked against: Character: Neville is nervous but determined, ready to support Luna and the plan they have put in place.
    quote: 'Let’s make sure we’re ready for anything'
No Contradiction: NA
Contradicted round: (non

### 7. S — H1 metrics

In [203]:
print(format_h1_metrics(h1_rows["S"]))

Round 1
Context Use: PASS
Elicited fact: Neville is nervous but determined, ready to support Luna and the plan they have put in place. Response: 'let’s make sure we’re ready for anything. I’ve got my gillyweed and a few other bits just in case. We can’t afford to be caught off guard, not tonight.'
World Consistency: PASS
  [consistent] they should be prepared for any situation
    checked against: Character: Neville is nervous but determined, ready to support Luna and the plan they have put in place.
    quote: 'let’s make sure we’re ready for anything'
  [consistent] Neville has gillyweed and other items for preparation
    checked against: No supplied fact addresses this claim.
    quote: 'I’ve got my gillyweed and a few other bits just in case'
  [consistent] they cannot afford to be caught off guard tonight
    checked against: Character: Neville is nervous but determined, ready to support Luna and the plan they have put in place.
    quote: 'We can’t afford to be caught off guard,

### 8. P — H1 metrics

In [204]:
print(format_h1_metrics(h1_rows["P"]))

Round 1
Context Use: PASS
Elicited fact: Current State = Neville is nervous but determined. Response: 'I just hope everything goes smoothly. I mean, it’s Harry, but still… we can’t be too careful.'
World Consistency: PASS
  [consistent] midnight is not too far off
    checked against: No supplied fact addresses this claim.
    quote: 'Midnight? That’s... that’s not too far off.'
  [consistent] they should be ready for anything
    checked against: No supplied fact addresses this claim.
    quote: 'We should make sure we’re ready for anything.'
  [consistent] Neville hopes everything goes smoothly
    checked against: Character: Current State = Neville is nervous but determined
    quote: 'I just hope everything goes smoothly.'
  [consistent] they can't be too careful even though it's Harry
    checked against: No supplied fact addresses this claim.
    quote: 'I mean, it’s Harry, but still… we can’t be too careful.'
No Contradiction: NA
Contradicted round: (none)
(none — N/A)

Round 2


### 9. U — H2 realism

H2 uses CharacterBox's two-call judge, so the judge's reasoning is visible in two places:

1. **Critique** — written first, in a separate call that never gives scores. This is the
   closest thing to "what the LLM thinks before scoring": it lists issues it found.
2. **Scores + explanation** — a second call receives the critique and returns the seven
   1–5 scores. Its explanation comes *after* the numbers in the same response, so it is a
   justification of scores already chosen, not reasoning that produced them.

When a score looks wrong, check the critique first: a mistaken critique (e.g. an invented
"fact" about the character) carries straight into the scores.

In [205]:
H2_METRICS = [
    "Knowledge Accuracy",
    "Emotional Expression",
    "Personality Traits",
    "Behavioral Accuracy",
    "Immersion",
    "Adaptability",
    "Behavioral Coherence",
]


def split_score_response(judge_response):
    """The score call answers "Metric: N" lines first, then (optionally) explains them.
    Everything after the last metric line is the explanation."""
    lines = judge_response.splitlines()
    last_score_line = -1
    for index, line in enumerate(lines):
        if any(line.strip().strip("*").startswith(f"{metric}:") for metric in H2_METRICS):
            last_score_line = index
    explanation = "\n".join(lines[last_score_line + 1:]).strip()
    return explanation or "(the judge gave scores only, no explanation)"


def format_h2(row):
    if row is None:
        return "(no H2 row for this condition yet — run evaluate.py on this generation run)"
    lines = [f"Judge: {row.get('judge_model')}  seed: {row.get('judge_seed')}"]
    lines.append(f"Rounds evaluated: {row.get('rounds_evaluated')}")
    lines.append("")
    lines.append("[Scores]")
    for metric in H2_METRICS:
        lines.append(f"  {metric:<22} {row['scores'].get(metric)}")
    lines.append(f"  {'Average':<22} {row['average']:.2f}")
    lines.append("")
    lines.append("[Critique — written before scoring, separate call]")
    lines.append(row["critique"].strip())
    lines.append("")
    lines.append("[Score explanation — written after the scores, same call]")
    lines.append(split_score_response(row["judge_response"]))
    return "\n".join(lines)


print(format_h2(h2_rows["U"]))

Judge: gpt-4o  seed: 1
Rounds evaluated: 10

[Scores]
  Knowledge Accuracy     3
  Emotional Expression   2
  Personality Traits     3
  Behavioral Accuracy    3
  Immersion              2
  Adaptability           2
  Behavioral Coherence   2
  Average                2.43

[Critique — written before scoring, separate call]
The role-play scenario presents several issues based on the evaluation criteria. Firstly, there is a factual inaccuracy regarding the signal to meet Harry; Luna initially states it as three taps on the window, but later it is referred to as two knocks on the door, creating inconsistency. Character consistency is compromised as Neville repeatedly emphasizes the need for quiet, which becomes redundant and detracts from engagement, suggesting a lack of adaptability in his interactions. This repetition also affects logical coherence, as Luna continues to speak loudly despite Neville's constant reminders, which seems unnatural. Emotional expression is limited, with Nevill

### 10. S — H2 realism

In [206]:
print(format_h2(h2_rows["S"]))

Judge: gpt-4o  seed: 1
Rounds evaluated: 10

[Scores]
  Knowledge Accuracy     1
  Emotional Expression   2
  Personality Traits     2
  Behavioral Accuracy    2
  Immersion              2
  Adaptability           2
  Behavioral Coherence   2
  Average                1.86

[Critique — written before scoring, separate call]
The role-play scenario presents several issues across the evaluation criteria. Firstly, there is a factual inaccuracy regarding Neville using the Invisibility Cloak, which is typically associated with Harry Potter, not Neville. Character consistency is compromised as Neville's repeated reminders to Luna to keep her voice down seem excessive and out of character for someone who is supposed to be brave and ready for action. This repetition also contributes to content redundancy, making the dialogue less engaging. Logical coherence is affected by Luna's repeated questions about the meeting details, which she should already know, suggesting a lack of logical progression 

### 11. P — H2 realism

Followed by a side-by-side score table for all three conditions.

In [207]:
print(format_h2(h2_rows["P"]))

print()
print("[U/S/P score comparison]")
print(f"  {'Metric':<22} " + "  ".join(f"{c:>4}" for c in CONDITION_ORDER))
for metric in H2_METRICS + ["Average"]:
    values = []
    for condition in CONDITION_ORDER:
        row = h2_rows[condition]
        if row is None:
            values.append("   -")
        elif metric == "Average":
            values.append(f"{row['average']:4.2f}")
        else:
            values.append(f"{row['scores'].get(metric):>4}")
    print(f"  {metric:<22} " + "  ".join(values))

Judge: gpt-4o  seed: 1
Rounds evaluated: 10

[Scores]
  Knowledge Accuracy     1
  Emotional Expression   2
  Personality Traits     2
  Behavioral Accuracy    2
  Immersion              2
  Adaptability           2
  Behavioral Coherence   2
  Average                1.86

[Critique — written before scoring, separate call]
The role-play scenario presents several issues based on the evaluation criteria. Firstly, there is a factual inaccuracy regarding Neville's use of the Invisibility Cloak, which is typically associated with Harry, not Neville. Character consistency is compromised as Neville's dialogue often portrays him as overly anxious and repetitive, which detracts from his established bravery and loyalty. Logical coherence is disrupted by Luna's repeated questioning about the meeting details, which seems unnatural given her usual attentiveness and intelligence. Content redundancy is evident in the repeated emphasis on the meeting time, location, and signal, which becomes monotonou